# Decorators & Generators

Two of Python's most powerful features for writing reusable, expressive, and memory-efficient code.

**In this notebook:**
- Basic decorator pattern
- `functools.wraps`
- Decorators with arguments
- Class-based decorators
- Stacking decorators
- `yield` and generator functions
- Generator expressions vs list comprehensions
- `send()` — two-way communication
- `yield from` — delegation
- Infinite generators and pipelines

## 1. Basic Decorator

A decorator wraps a function to add behaviour before and/or after it runs.

`@my_decorator` above a function is syntactic sugar for `func = my_decorator(func)`.

In [ ]:
def my_decorator(func):
    def wrapper(*args, **kwargs):
        print(f'→ Calling {func.__name__}')
        result = func(*args, **kwargs)
        print(f'← Done')
        return result
    return wrapper

@my_decorator
def greet(name):
    print(f'Hello, {name}!')

greet('Alice')
# Equivalent to: greet = my_decorator(greet); greet('Alice')

## 2. functools.wraps

Without `@wraps`, the wrapper replaces `__name__` and `__doc__`. Always use it.

In [ ]:
import functools
import time

def timer(func):
    @functools.wraps(func)   # preserves __name__, __doc__, __module__
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = func(*args, **kwargs)
        print(f'{func.__name__} took {time.perf_counter()-start:.4f}s')
        return result
    return wrapper

@timer
def slow_sum(n):
    """Return sum of 0..n."""
    return sum(range(n))

print(slow_sum(1_000_000))
print(slow_sum.__name__)   # slow_sum (not 'wrapper')
print(slow_sum.__doc__)    # Return sum of 0..n.

## 3. Decorators with Arguments

Add an outer function that receives the arguments — three levels of nesting.

In [ ]:
def repeat(times):
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def say(msg):
    print(msg)

say('Hello!')
# Hello!
# Hello!
# Hello!

## 4. Class-based Decorator

Use a class with `__call__` when the decorator needs to maintain state between calls.

In [ ]:
class CountCalls:
    def __init__(self, func):
        functools.update_wrapper(self, func)
        self.func  = func
        self.count = 0

    def __call__(self, *args, **kwargs):
        self.count += 1
        print(f'{self.func.__name__} called {self.count} time(s)')
        return self.func(*args, **kwargs)

@CountCalls
def add(a, b):
    return a + b

add(1, 2)
add(3, 4)
print(f'Total calls: {add.count}')   # 2

## 5. Stacking Decorators

Decorators apply **bottom-up** — the one closest to the function runs first.

In [ ]:
def bold(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return f'**{func(*args, **kwargs)}**'
    return wrapper

def upper(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper

@bold    # applied second (outer)
@upper   # applied first (inner)
def greet(name):
    return f'hello, {name}'

print(greet('alice'))   # **HELLO, ALICE**

# Equivalent to:
# greet = bold(upper(greet))

## 6. yield — Generator Basics

A generator function uses `yield` to pause and produce values one at a time. It maintains state between calls.

In [ ]:
def countdown(n):
    print('Starting countdown')
    while n > 0:
        yield n          # pause, produce n
        n -= 1
    print('Done!')

gen = countdown(3)
print(next(gen))   # Starting countdown → 3
print(next(gen))   # 2
print(next(gen))   # 1
# next(gen) → StopIteration (generator exhausted)

# for loop handles StopIteration automatically
for n in countdown(5):
    print(n, end=' ')   # 5 4 3 2 1

## 7. Generator Expressions vs List Comprehensions

Generator expressions are lazy — they compute values on demand and use constant memory.

In [ ]:
import sys

N = 100_000

list_comp = [n ** 2 for n in range(N)]   # all values in memory
gen_expr  = (n ** 2 for n in range(N))   # lazy, one at a time

print(f'List size: {sys.getsizeof(list_comp):,} bytes')
print(f'Gen size:  {sys.getsizeof(gen_expr):,} bytes')

# sum() works with generators — never builds the full list
total = sum(n ** 2 for n in range(1_000_000))
print(f'Sum of squares: {total:,}')

# next() pulls one value from a generator
first_even_square = next(n**2 for n in range(100) if n % 2 == 0 and n > 0)
print(f'First even square > 0: {first_even_square}')   # 4

## 8. send() — Two-way Communication

`send(value)` resumes the generator AND injects a value. The `yield` expression evaluates to it.

In [ ]:
def running_average():
    total, count = 0, 0
    while True:
        value = yield (total / count if count else 0.0)
        if value is None:
            return
        total += value
        count += 1

gen = running_average()
next(gen)              # prime — advance to first yield
print(gen.send(10))    # 10.0
print(gen.send(20))    # 15.0
print(gen.send(30))    # 20.0
print(gen.send(40))    # 25.0

## 9. yield from — Delegation

`yield from` delegates to a sub-generator, transparently forwarding `next()`, `send()`, and exceptions.

In [ ]:
def flatten(nested):
    for item in nested:
        if isinstance(item, list):
            yield from flatten(item)   # delegate recursively
        else:
            yield item

data = [1, [2, [3, 4], 5], [6, 7]]
print(list(flatten(data)))   # [1, 2, 3, 4, 5, 6, 7]

# yield from also works with any iterable
def chain(*iterables):
    for it in iterables:
        yield from it

print(list(chain([1, 2], [3, 4], [5])))   # [1, 2, 3, 4, 5]

## 10. Infinite Generators & Pipelines

Generators can run forever — pull only what you need. Chain them to build lazy pipelines.

In [ ]:
def fibonacci():
    a, b = 0, 1
    while True:
        yield a
        a, b = b, a + b

def take(n, iterable):
    for i, val in enumerate(iterable):
        if i >= n: return
        yield val

def only_even(iterable):
    for val in iterable:
        if val % 2 == 0:
            yield val

# Pipeline: fibonacci → filter evens → take 5
pipeline = take(5, only_even(fibonacci()))
print(list(pipeline))   # [0, 2, 8, 34, 144]

## Practice

| File | Difficulty | Topics |
|---|---|---|
| [01-easy.py](exercises/01-easy.py) | Easy | timer, logger, countdown, fibonacci, take |
| [02-medium.py](exercises/02-medium.py) | Medium | retry, memoize, pipeline, running_average, stacking |
| [03-challenge.py](exercises/03-challenge.py) | Challenge | RateLimit class decorator, deep_flatten, coroutine pipeline |

Solutions: [solutions/](solutions/)